1. Retail Dataset For Meaningful Insights

In [1]:
# Import Libraries

import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

In [2]:
#1. then we bring the dataset

retail_sales_dataset = pd.read_csv('/content/retail_sales_dataset.csv')

FileNotFoundError: [Errno 2] No such file or directory: '/content/retail_sales_dataset.csv'

In [ ]:
retail_sales_dataset

In [ ]:
# Cleaning the dataset

retail_sales_dataset.isnull().sum()

In [ ]:
# check redundant rows

retail_sales_dataset.duplicated().sum()

In [ ]:
#2. Then We move to Descriptive Statistics Calculate basic statistics (mean, median, mode, standard deviation).

print("Columns in the dataset:")
retail_sales_dataset_columns = retail_sales_dataset.columns
print(retail_sales_dataset_columns)

print("\nDescriptive statistics for each column:")

for column in retail_sales_dataset_columns:

    print(f"\n--- Statistics for column: {column} ---")
    if pd.api.types.is_numeric_dtype(retail_sales_dataset[column]):
        print(f"Count: {retail_sales_dataset[column].count()}")
        print(f"Mean: {retail_sales_dataset[column].mean():.2f}")
        print(f"Median: {retail_sales_dataset[column].median():.2f}")
        print(f"Standard Deviation: {retail_sales_dataset[column].std():.2f}")

    else:
        # non - numeric
        print(retail_sales_dataset[column].describe())
        print("Mode:")
        # .mode() returns a Series, take the first element if not empty
        print(retail_sales_dataset[column].mode().iloc[0] if not retail_sales_dataset[column].mode().empty else "No mode found")

In [ ]:
#3. Time Series Analysis: Analyze sales trends over time using time series techniques.

# Ensure 'Date' column is datetime type
retail_sales_dataset['Date'] = pd.to_datetime(retail_sales_dataset['Date'])

# Sort the dataset by Date for time series analysis
retail_sales_dataset_sorted = retail_sales_dataset.sort_values(by='Date')

# Aggregate total sales per day
daily_sales = retail_sales_dataset_sorted.groupby('Date')['Total Amount'].sum().reset_index()

# Create the line chart using seaborn
plt.figure(figsize=(12, 6))
sns.lineplot(x='Date', y='Total Amount', data=daily_sales)
plt.title('Daily Total Sales Over Time')
plt.xlabel('Date')
plt.ylabel('Total Sales')
plt.grid(True)
plt.tight_layout()
plt.show()

In [ ]:
#4. Customer and Product Analysis: Analyze customer demographics and purchasing behavior.

# first analyze using which gender by doing much transactions ( GROUP BY ) and also total quantity

retail_sales_dataset_gender = retail_sales_dataset.groupby('Gender').agg(
    total_amount=('Total Amount', 'sum'),
    total_quantity=('Quantity', 'sum')
).reset_index()

retail_sales_dataset_gender

Here We Can Say That **Female** has much Transactions amount

In [ ]:
# now we find according to age category ( Young , Adult , Middle-aged , Seniors )

# Define age bins and labels based on the user's request
# Ages below 18 will not be categorized, as the ranges start from 18.
bins = [17, 25, 35, 50, retail_sales_dataset['Age'].max() + 1]
labels = ['Young Adults', 'Adults', 'Middle-aged', 'Seniors']

# Create 'Age Category' column. `right=True` means (start, end]
# This categorizes ages as follows:
# (17, 25] -> 18-25 (Young Adults)
# (25, 35] -> 26-35 (Adults)
# (35, 50] -> 36-50 (Middle-aged)
# (50, max_age] -> 51+ (Seniors)
retail_sales_dataset['Age Category'] = pd.cut(
    retail_sales_dataset['Age'],
    bins=bins,
    labels=labels,
    right=True,
    include_lowest=False
)

# Group by 'Age Category' and calculate total amount and total quantity
retail_sales_dataset_age_category = retail_sales_dataset.groupby('Age Category').agg(
    total_amount=('Total Amount', 'sum'),
    total_quantity=('Quantity', 'sum')
).reset_index()

# Create a dictionary to map Age Category to its range string
age_range_map = {
    'Young Adults': '18->25',
    'Adults': '26->35',
    'Middle-aged': '36->50',
    'Seniors': f'51->{int(retail_sales_dataset['Age'].max())}'
}
retail_sales_dataset_age_category['Age_range'] = retail_sales_dataset_age_category['Age Category'].map(age_range_map)

# Reorder columns to place 'Age_range' after 'Age Category'
retail_sales_dataset_age_category = retail_sales_dataset_age_category[['Age Category', 'Age_range', 'total_amount', 'total_quantity']]

display(retail_sales_dataset_age_category)

In [ ]:
# now analyze customer behaviour ( which product category buys most & no of products that bought )


retail_sales_dataset_product_category = retail_sales_dataset.groupby('Product Category').agg(
    total_amount=('Total Amount', 'sum'),
    total_quantity=('Quantity', 'sum')
).reset_index()

retail_sales_dataset_product_category

In [ ]:
#5. Visualization: Present insights through bar charts, line plots, and heatmaps.

# first using bar charts of each product cateogry which bought most

plt.figure(figsize=(10, 6))
sns.barplot(x='Product Category', y='total_quantity', data=retail_sales_dataset_product_category, palette='viridis')
plt.title('Total Quantity of Items Sold by Product Category')
plt.xlabel('Product Category')
plt.ylabel('Total Quantity Sold')
plt.grid(axis='y', linestyle='--')
plt.tight_layout()
plt.show()

In [ ]:
# line charts analysis

plt.figure(figsize=(12, 6))
sns.lineplot(x='Date', y='Total Amount', data=daily_sales, marker='o', color='blue')
plt.title('Daily Total Sales Over Time')
plt.xlabel('Date')
plt.ylabel('Total Sales')
plt.grid(True)
plt.tight_layout()
plt.show()


# for particular month between the 2023-05 to 2023-07

# Filter daily_sales for the specified date range
monthly_sales_2023_05_to_07 = daily_sales[
    (daily_sales['Date'] >= '2023-05-01') & (daily_sales['Date'] <= '2023-07-31')
]

plt.figure(figsize=(12, 6))
sns.lineplot(x='Date', y='Total Amount', data=monthly_sales_2023_05_to_07, marker='o', color='green')
plt.title('Daily Total Sales from May to July 2023')
plt.xlabel('Date')
plt.ylabel('Total Sales')
plt.grid(True)
plt.tight_layout()
plt.show()

In [ ]:
# heatmap ( between the attributes correlations )

# Select only numeric columns for correlation calculation
numeric_retail_sales_dataset = retail_sales_dataset.select_dtypes(include=['number'])

correlation_matrix = numeric_retail_sales_dataset.corr()

plt.figure(figsize=(10, 8))
sns.heatmap(correlation_matrix, annot=True, cmap='coolwarm', fmt=".2f")
plt.title('Correlation Heatmap')
plt.show()

# #6. Recommendations

if we want to increase the sales data then we want to add more